In [8]:
import sys
sys.path.append('..')

import torch 
import torch.nn as nn
import torch.optim as optim
from src.dataset import get_dataloaders
from src.model import get_model
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path('../data/raw/PlantVillage')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

cpu


In [9]:
train_data, val_data, test_data, classes = get_dataloaders(DATA_DIR, batch_size = 32)

In [11]:
print(f"Train Data : {train_data}")
print(f"Test Data : {test_data}")
print(f"Val Data : {val_data}")
print(f"Classes : {classes}")

Train Data : <torch.utils.data.dataloader.DataLoader object at 0x00000244377E4220>
Test Data : <torch.utils.data.dataloader.DataLoader object at 0x000002444AFFEEF0>
Val Data : <torch.utils.data.dataloader.DataLoader object at 0x0000024436441D20>
Classes : ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [12]:
model_0 = get_model(num_classes =15)

In [13]:
model_0 = model_0.to(DEVICE)

In [19]:
print(model_0.parameters())


<generator object Module.parameters at 0x000002444B70DE00>


In [20]:
class_counts = {cls: len(list((DATA_DIR / cls).glob('*.jpg'))) for cls in classes}
total = sum(class_counts.values())
weights = [total / (len(classes) * class_counts[cls]) for cls in classes]
weights_tensor = torch.tensor(weights, dtype=torch.float)
print(weights_tensor)


tensor([1.3799, 0.9314, 1.3757, 1.3757, 9.0509, 0.6468, 1.3757, 0.7210, 1.4451,
        0.7768, 0.8208, 0.9799, 0.4288, 3.6883, 0.8647])


In [24]:
criterion = nn.CrossEntropyLoss(weight = weights_tensor)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model_0.parameters()), lr=0.001)

In [25]:
print(model_0)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [79]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    #Training
    model.train()

    #loss calculation metric
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:        # loop ALL batches
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()        # += not =+
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total * 100
    return avg_loss, accuracy


    

In [82]:
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            val_output = model(images)
            loss = criterion(val_output, labels)
            
            total_loss += loss.item()
            
            val_preds = torch.argmax(val_output, dim=1)
            correct += (val_preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total * 100
    return avg_loss, accuracy

In [ ]:
EPOCHS = 10
best_val_acc = 0.0

train_losses = [] 
val_losses = []
train_accuracies = []
val_accuracies = []


for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model_0, train_data, criterion, optimizer, DEVICE)

    v_loss, v_acc = validate(model_0, val_data, criterion, DEVICE)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    train_accuracies.append(t_acc)
    val_accuracies.append(v_acc)

    print(f"Epoch {epoch+1:02d}/10 | Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.2f}% | Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.2f}%")

if v_acc > best_val_acc:
    best_val_acc = v_acc
    torch.save(model_0.state_dict(), '../models/best_model.pth')
    print(f"  → Best model saved!")

Epoch 01/10 | Train Loss: 0.3436 | Train Acc: 89.46% | Val Loss: 0.2349 | Val Acc: 93.94%
Epoch 02/10 | Train Loss: 0.2842 | Train Acc: 91.11% | Val Loss: 0.2029 | Val Acc: 94.09%
